In [ ]:
# ============================================================
# 0. Imports and settings
# ============================================================

from pathlib import Path
import warnings
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.compose import TransformedTargetRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

warnings.filterwarnings("ignore")
DATA_PATH = Path("df_base.csv")

HOLDOUT_SEEDS = list(range(10))
TEST_SIZE = 0.20

PCE_TARGET = "jv.default_PCE"
COMPONENT_TARGETS = ["jv.default_Jsc", "jv.default_Voc", "jv.default_FF"]
ALL_TARGET_COLS = COMPONENT_TARGETS + [PCE_TARGET]

print("DATA_PATH:", DATA_PATH.resolve())
print("HOLDOUT_SEEDS:", HOLDOUT_SEEDS)

DATA_PATH: /Users/rosylu/Desktop/PVK/new/df_base''.csv
HOLDOUT_SEEDS: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [ ]:
# ============================================================
# 1. Load df_base and create the exact Level-3 features allowed for selected PCE
# ============================================================

df_base = pd.read_csv("df_base.csv", low_memory=False)

for c in [
    "PVK_thickness_clean", "ETL_thickness_clean", "HTL_thickness_clean", "BC_thickness_clean",
    "perovskite_bandgap_clean", "jv.default_Jsc", "jv.default_Voc", "jv.default_FF", "jv.default_PCE",
    "ETL_n_layers", "HTL_n_layers", "BC_n_layers",
]:
    if c in df_base.columns:
        df_base[c] = pd.to_numeric(df_base[c], errors="coerce")


def num_col(d, col):
    if col in d.columns:
        return pd.to_numeric(d[col], errors="coerce")
    return pd.Series(np.nan, index=d.index)


def safe_div(a, b):
    return (a / b.replace(0, np.nan)).replace([np.inf, -np.inf], np.nan)


def add_selected_level3_features(d):
    """
    Same policy as PCE_one_stage_vs_two.ipynb:
    - Absorber_to_ETL_ratio is retained from the Jsc/Voc selected feature set.
    - ff_PVK_fraction_of_total_stack is the single retained FF Level-3 feature.
    """
    d = d.copy()
    pvk = num_col(d, "PVK_thickness_clean")
    etl = num_col(d, "ETL_thickness_clean")
    htl = num_col(d, "HTL_thickness_clean")
    bc = num_col(d, "BC_thickness_clean")

    d["Absorber_to_ETL_ratio"] = safe_div(pvk, etl)
    d["ff_PVK_fraction_of_total_stack"] = safe_div(pvk, pvk + etl + htl + bc)
    return d


df_base = add_selected_level3_features(df_base)
df_base = df_base.replace([np.inf, -np.inf], np.nan)

LEVEL3_FEATURES_ALLOWED_IN_SELECTED = [
    "Absorber_to_ETL_ratio",
    "ff_PVK_fraction_of_total_stack",
]

print("Rows in df_base:", len(df_base))
print("Allowed selected Level-3 features present:", [c for c in LEVEL3_FEATURES_ALLOWED_IN_SELECTED if c in df_base.columns])

Rows in df_base: 3182
Allowed selected Level-3 features present: ['Absorber_to_ETL_ratio', 'ff_PVK_fraction_of_total_stack']


In [3]:
# ============================================================
# 2. Read the frozen selected feature definitions from notebook 04 outputs
# ============================================================

FEATURE_DIR_CANDIDATES = [
    Path("component_feature_selection_outputs"),
    Path("."),
    Path("project_sources/component_feature_selection_outputs"),
]


def resolve_feature_file(filename):
    candidates = [directory / filename for directory in FEATURE_DIR_CANDIDATES]
    path = next((candidate for candidate in candidates if candidate.exists()), None)
    if path is None:
        raise FileNotFoundError(
            f"Could not find {filename}. Checked: "
            + ", ".join(map(str, candidates))
        )
    return path


def read_selected_features(path, expected_count):
    table = pd.read_csv(path)
    if "selected_feature" not in table.columns:
        raise ValueError(f"{path} must contain a 'selected_feature' column")

    if table["selected_feature"].isna().any():
        raise ValueError(f"{path} contains missing feature names")

    features = table["selected_feature"].astype(str).str.strip().tolist()
    if any(not feature for feature in features):
        raise ValueError(f"{path} contains blank feature names")
    if len(features) != expected_count:
        raise ValueError(
            f"{path} must contain {expected_count} features; got {len(features)}"
        )
    if len(features) != len(set(features)):
        raise ValueError(f"{path} contains duplicate feature names")
    return features


JSC_FEATURE_PATH = resolve_feature_file("jsc_selected_top20.csv")
VOC_FEATURE_PATH = resolve_feature_file("voc_selected_top20.csv")
FF_FEATURE_PATH = resolve_feature_file("ff_selected_top30.csv")

JSC_SELECTED_FEATURES = read_selected_features(JSC_FEATURE_PATH, 20)
VOC_SELECTED_FEATURES = read_selected_features(VOC_FEATURE_PATH, 20)
FF_SELECTED_FEATURES = read_selected_features(FF_FEATURE_PATH, 30)


def unique_keep_order(cols):
    out = []
    for col in cols:
        if col not in out:
            out.append(col)
    return out


PCE_SELECTED_FEATURES_FROZEN = unique_keep_order(
    JSC_SELECTED_FEATURES + VOC_SELECTED_FEATURES + FF_SELECTED_FEATURES
)

EXPECTED_PCE_UNION_COUNT = 37
assert len(PCE_SELECTED_FEATURES_FROZEN) == EXPECTED_PCE_UNION_COUNT, (
    f"Expected a {EXPECTED_PCE_UNION_COUNT}-feature Jsc20/Voc20/FF30 union; "
    f"got {len(PCE_SELECTED_FEATURES_FROZEN)}. Check that the CSVs are the "
    "latest outputs from notebook 04."
)

print("Jsc feature file:", JSC_FEATURE_PATH.resolve())
print("Voc feature file:", VOC_FEATURE_PATH.resolve())
print("FF feature file:", FF_FEATURE_PATH.resolve())
print("Frozen component counts:", {"Jsc": 20, "Voc": 20, "FF": 30})
print("Frozen PCE selected union:", len(PCE_SELECTED_FEATURES_FROZEN))


Jsc feature file: /Users/rosylu/Desktop/PVK/new/component_feature_selection_outputs/jsc_selected_top20.csv
Voc feature file: /Users/rosylu/Desktop/PVK/new/component_feature_selection_outputs/voc_selected_top20.csv
FF feature file: /Users/rosylu/Desktop/PVK/new/component_feature_selection_outputs/ff_selected_top30.csv
Frozen component counts: {'Jsc': 20, 'Voc': 20, 'FF': 30}
Frozen PCE selected union: 37


In [4]:
# ============================================================
# 3. Build the final selected-37 PCE matrix
# ============================================================

missing_targets = [c for c in ALL_TARGET_COLS if c not in df_base.columns]
if missing_targets:
    raise ValueError("Missing target columns: " + ", ".join(missing_targets))

Y_all = df_base[ALL_TARGET_COLS].copy()
for col in ALL_TARGET_COLS:
    Y_all[col] = pd.to_numeric(Y_all[col], errors="coerce")

# Same PCE-aligned sample policy as the previous kernel notebook:
# keep rows where PCE and all component targets are available.
mask = Y_all[ALL_TARGET_COLS].notna().all(axis=1)
y_pce = Y_all.loc[mask, PCE_TARGET].reset_index(drop=True)

X_source = (
    df_base.loc[mask]
    .copy()
    .replace({pd.NA: np.nan})
    .reset_index(drop=True)
)


def present_and_missing(cols, X):
    cols = unique_keep_order(cols)
    present = [c for c in cols if c in X.columns]
    missing = [c for c in cols if c not in X.columns]
    return present, missing


JSC_SELECTED_PRESENT, JSC_SELECTED_MISSING = present_and_missing(
    JSC_SELECTED_FEATURES, X_source
)
VOC_SELECTED_PRESENT, VOC_SELECTED_MISSING = present_and_missing(
    VOC_SELECTED_FEATURES, X_source
)
FF_SELECTED_PRESENT, FF_SELECTED_MISSING = present_and_missing(
    FF_SELECTED_FEATURES, X_source
)

missing_selected = sorted(set(
    JSC_SELECTED_MISSING + VOC_SELECTED_MISSING + FF_SELECTED_MISSING
))
if missing_selected:
    raise ValueError(
        "Final selected features are missing from df_base: "
        + ", ".join(missing_selected)
    )

PCE_SELECTED_FEATURES = unique_keep_order(
    JSC_SELECTED_PRESENT + VOC_SELECTED_PRESENT + FF_SELECTED_PRESENT
)
X_pce_selected = X_source[PCE_SELECTED_FEATURES].copy()

assert len(JSC_SELECTED_PRESENT) == 20
assert len(VOC_SELECTED_PRESENT) == 20
assert len(FF_SELECTED_PRESENT) == 30
assert len(PCE_SELECTED_FEATURES) == EXPECTED_PCE_UNION_COUNT

feature_audit = pd.DataFrame({
    "set": ["Jsc selected", "Voc selected", "FF selected", "PCE selected union"],
    "n_present": [
        len(JSC_SELECTED_PRESENT),
        len(VOC_SELECTED_PRESENT),
        len(FF_SELECTED_PRESENT),
        len(PCE_SELECTED_FEATURES),
    ],
    "n_missing": [
        len(JSC_SELECTED_MISSING),
        len(VOC_SELECTED_MISSING),
        len(FF_SELECTED_MISSING),
        0,
    ],
    "missing_features": [
        "; ".join(JSC_SELECTED_MISSING),
        "; ".join(VOC_SELECTED_MISSING),
        "; ".join(FF_SELECTED_MISSING),
        "",
    ],
})

display(feature_audit)

print("PCE-aligned samples:", len(X_pce_selected))
print("PCE selected raw features:", X_pce_selected.shape[1])
print(
    "Allowed Level-3 in selected PCE:",
    [c for c in LEVEL3_FEATURES_ALLOWED_IN_SELECTED if c in X_pce_selected.columns],
)
print(f"\nPCE_SELECTED_FEATURES ({len(PCE_SELECTED_FEATURES)}):")
for number, feature in enumerate(PCE_SELECTED_FEATURES, start=1):
    print(f" {number:>2}. {feature}")


,set,n_present,n_missing,missing_features
0,Jsc selected,20,0,
1,Voc selected,20,0,
2,FF selected,30,0,
3,PCE selected union,37,0,


PCE-aligned samples: 3182
PCE selected raw features: 37
Allowed Level-3 in selected PCE: ['Absorber_to_ETL_ratio', 'ff_PVK_fraction_of_total_stack']

PCE_SELECTED_FEATURES (37):
  1. perovskite_bandgap_clean
  2. Absorber_to_ETL_ratio
  3. PVK_thickness_clean
  4. perovskite_deposition.quenching_induced_crystallisation
  5. HTL_thickness_clean
  6. etl.deposition_procedure
  7. htl.stack_sequence
  8. perovskite.composition_short_form
  9. cell.area_measured
 10. etl.stack_sequence
 11. backcontact.stack_sequence
 12. ETL_thickness_clean
 13. perovskite_deposition.synthesis_atmosphere
 14. perovskite_deposition.thermal_annealing_time
 15. BC_thickness_clean
 16. perovskite_deposition.procedure
 17. perovskite_deposition.thermal_annealing_temperature
 18. htl.deposition_procedure
 19. jv.light_masked_cell
 20. jv.test_atmosphere
 21. perovskite.composition_b_ions
 22. perovskite_deposition.solvents
 23. perovskite.composition_leadfree
 24. perovskite_deposition.quenching_media
 25. pero

In [5]:
# ============================================================
# 4. Model helpers: same features, model-specific preprocessing where needed
# ============================================================

def make_onehot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def make_preprocessor(X, scale_numeric=False):
    num_cols = X.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X.columns if c not in num_cols]

    if scale_numeric:
        numeric_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ])
    else:
        numeric_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
        ])

    categorical_transformer = Pipeline([
            ("to_object", FunctionTransformer(
                lambda x: x.astype(object), validate=False
            )),
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("to_string", FunctionTransformer(lambda x: x.astype(str), validate=False)),
            ("onehot", make_onehot_encoder()),
        ])

    return ColumnTransformer([
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ])


def rmse_score(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))


def count_encoded_features_after_fit(pipeline):
    prep = pipeline.named_steps["prep"]
    try:
        num_cols = prep.transformers_[0][2]
        cat_cols = prep.transformers_[1][2]
        n_num = len(num_cols)
        if len(cat_cols) == 0:
            return int(n_num)
        cat_pipe = prep.named_transformers_["cat"]
        onehot = cat_pipe.named_steps["onehot"]
        n_cat = int(sum(len(cats) for cats in onehot.categories_))
        return int(n_num + n_cat)
    except Exception:
        return np.nan

In [6]:
# ============================================================
# 5. Custom block-kernel model
# ============================================================

from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import pairwise_distances
from sklearn.model_selection import KFold, RandomizedSearchCV


class PCEBlockKernelRidge(BaseEstimator, RegressorMixin):
    """Kernel Ridge model with physics-guided PCE component block kernels.

    Blocks:
    - Jsc selected features
    - Voc selected features
    - FF selected features

    Combination:
    - additive: weighted sum of block kernels
    - product: product of block kernels

    Gamma is parameterised by a multiplier over the train-fold median squared
    distance for each block. This keeps gamma values comparable across blocks
    with different encoded dimensionality.
    """

    def __init__(
        self,
        block_features=None,
        combination="additive",
        alpha=1.0,
        gamma_jsc_mult=1.0,
        gamma_voc_mult=1.0,
        gamma_ff_mult=1.0,
        additive_weights=(1.0, 1.0, 1.0),
    ):
        self.block_features = block_features
        self.combination = combination
        self.alpha = alpha
        self.gamma_jsc_mult = gamma_jsc_mult
        self.gamma_voc_mult = gamma_voc_mult
        self.gamma_ff_mult = gamma_ff_mult
        self.additive_weights = additive_weights

    def _make_block_preprocessor(self, X_block):
        num_cols = X_block.select_dtypes(include=np.number).columns.tolist()
        cat_cols = [c for c in X_block.columns if c not in num_cols]

        numeric_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ])

        categorical_transformer = Pipeline([
            ("to_object", FunctionTransformer(
                lambda x: x.astype(object), validate=False
            )),
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("to_string", FunctionTransformer(lambda x: x.astype(str), validate=False)),
            ("onehot", make_onehot_encoder()),
        ])

        return ColumnTransformer([
            ("num", numeric_transformer, num_cols),
            ("cat", categorical_transformer, cat_cols),
        ])

    @staticmethod
    def _median_gamma(Z, multiplier):
        D2 = pairwise_distances(Z, metric="sqeuclidean")
        positive = D2[D2 > 0]
        if positive.size == 0:
            return float(multiplier)
        median_d2 = np.median(positive)
        if not np.isfinite(median_d2) or median_d2 <= 0:
            return float(multiplier)
        return float(multiplier / median_d2)

    @staticmethod
    def _rbf_from_sqdist(D2, gamma):
        return np.exp(-gamma * D2)

    def _fit_transform_blocks(self, X):
        self.block_names_ = ["Jsc", "Voc", "FF"]
        self.preprocessors_ = {}
        self.Z_train_ = {}
        self.gamma_ = {}

        gamma_mults = {
            "Jsc": self.gamma_jsc_mult,
            "Voc": self.gamma_voc_mult,
            "FF": self.gamma_ff_mult,
        }

        for block in self.block_names_:
            features = self.block_features[block]
            X_block = X[features].copy()
            prep = self._make_block_preprocessor(X_block)
            Z = prep.fit_transform(X_block)
            self.preprocessors_[block] = prep
            self.Z_train_[block] = Z
            self.gamma_[block] = self._median_gamma(Z, gamma_mults[block])

    def _transform_block(self, X, block):
        features = self.block_features[block]
        X_block = X[features].copy()
        return self.preprocessors_[block].transform(X_block)

    def _combined_kernel_train(self):
        kernels = []
        for block in self.block_names_:
            Z = self.Z_train_[block]
            D2 = pairwise_distances(Z, metric="sqeuclidean")
            kernels.append(self._rbf_from_sqdist(D2, self.gamma_[block]))

        if self.combination == "additive":
            weights = np.asarray(self.additive_weights, dtype=float)
            weights = weights / weights.sum()
            K = sum(w * k for w, k in zip(weights, kernels))
        elif self.combination == "product":
            K = kernels[0] * kernels[1] * kernels[2]
        else:
            raise ValueError("combination must be 'additive' or 'product'")

        return K

    def _combined_kernel_test(self, X):
        kernels = []
        for block in self.block_names_:
            Z_test = self._transform_block(X, block)
            Z_train = self.Z_train_[block]
            D2 = pairwise_distances(Z_test, Z_train, metric="sqeuclidean")
            kernels.append(self._rbf_from_sqdist(D2, self.gamma_[block]))

        if self.combination == "additive":
            weights = np.asarray(self.additive_weights, dtype=float)
            weights = weights / weights.sum()
            K = sum(w * k for w, k in zip(weights, kernels))
        elif self.combination == "product":
            K = kernels[0] * kernels[1] * kernels[2]
        else:
            raise ValueError("combination must be 'additive' or 'product'")

        return K

    def fit(self, X, y):
        X = X.copy()
        y_array = np.asarray(y, dtype=float)

        self._fit_transform_blocks(X)
        K_train = self._combined_kernel_train()

        # KernelRidge with a precomputed kernel has no explicit intercept.
        # Centre the target within every training fold, then add the mean
        # back in predict(). This keeps additive/product comparisons fair.
        self.y_mean_ = float(np.mean(y_array))
        y_centered = y_array - self.y_mean_

        self.model_ = KernelRidge(alpha=self.alpha, kernel="precomputed")
        self.model_.fit(K_train, y_centered)
        return self

    def predict(self, X):
        K_test = self._combined_kernel_test(X.copy())
        return self.model_.predict(K_test) + self.y_mean_


In [7]:
# ============================================================
# 6. Build PCE component feature blocks
# ============================================================

BLOCK_FEATURES = {
    "Jsc": JSC_SELECTED_PRESENT,
    "Voc": VOC_SELECTED_PRESENT,
    "FF": FF_SELECTED_PRESENT,
}

block_audit = pd.DataFrame({
    "block": list(BLOCK_FEATURES.keys()),
    "n_features": [len(v) for v in BLOCK_FEATURES.values()],
    "features": ["; ".join(v) for v in BLOCK_FEATURES.values()],
})

display(block_audit)

# Use the union matrix as the container passed to the estimator.
# The estimator selects each block internally, so train/test splits stay aligned.
X_kernel_source = X_source.copy()
print("PCE samples:", len(y_pce))
print("PCE selected union features:", len(PCE_SELECTED_FEATURES))
print("Jsc/Voc/FF block sizes:", {k: len(v) for k, v in BLOCK_FEATURES.items()})


,block,n_features,features
0,Jsc,20,perovskite_bandgap_clean; Absorber_to_ETL_rati...
1,Voc,20,htl.stack_sequence; perovskite.composition_b_i...
2,FF,30,ff_PVK_fraction_of_total_stack; htl.stack_sequ...


PCE samples: 3182
PCE selected union features: 37
Jsc/Voc/FF block sizes: {'Jsc': 20, 'Voc': 20, 'FF': 30}


In [8]:
# ============================================================
# 7. Tune additive vs product PCE block kernels
# ============================================================

INNER_CV_SPLITS = 3
N_ITER_BLOCK_KERNEL = 16

COMMON_KERNEL_PARAMS = {
    "alpha": [0.1, 1.0, 10.0, 100.0],
    "gamma_jsc_mult": [0.3, 1.0, 3.0, 10.0],
    "gamma_voc_mult": [0.3, 1.0, 3.0, 10.0],
    "gamma_ff_mult": [0.3, 1.0, 3.0, 10.0],
}

ADDITIVE_PARAM_DISTRIBUTIONS = {
    **COMMON_KERNEL_PARAMS,
    "additive_weights": [
        (1.0, 1.0, 1.0),
        (2.0, 1.0, 1.0),
        (1.0, 2.0, 1.0),
        (1.0, 1.0, 0.5),
        (1.0, 1.0, 0.25),
        (2.0, 2.0, 0.5),
    ],
}

PRODUCT_PARAM_DISTRIBUTIONS = COMMON_KERNEL_PARAMS.copy()


def make_block_kernel_search(combination, split_seed):
    if combination == "additive":
        param_dist = ADDITIVE_PARAM_DISTRIBUTIONS
    elif combination == "product":
        param_dist = PRODUCT_PARAM_DISTRIBUTIONS
    else:
        raise ValueError(combination)

    base = PCEBlockKernelRidge(
        block_features=BLOCK_FEATURES,
        combination=combination,
    )

    inner_cv = KFold(n_splits=INNER_CV_SPLITS, shuffle=True, random_state=split_seed)
    return RandomizedSearchCV(
        estimator=base,
        param_distributions=param_dist,
        n_iter=N_ITER_BLOCK_KERNEL,
        scoring="neg_root_mean_squared_error",
        cv=inner_cv,
        random_state=split_seed,
        n_jobs=1,
        refit=True,
        return_train_score=False,
        verbose=0,
    )

In [9]:
# ============================================================
# 8. Repeated held-out evaluation
# ============================================================

kernel_rows = []

for split_seed in HOLDOUT_SEEDS:
    print("\n" + "=" * 100)
    print(f"PCE custom block kernel | held-out split seed {split_seed}")
    print("=" * 100)

    idx = np.arange(len(y_pce))
    train_idx, test_idx = train_test_split(
        idx,
        test_size=TEST_SIZE,
        random_state=split_seed,
        shuffle=True,
    )

    X_train = X_kernel_source.iloc[train_idx].reset_index(drop=True)
    X_test = X_kernel_source.iloc[test_idx].reset_index(drop=True)
    y_train = y_pce.iloc[train_idx].reset_index(drop=True)
    y_test = y_pce.iloc[test_idx].reset_index(drop=True)

    for combination, model_name in [
        ("additive", "PCE_component_additive_kernel"),
        ("product", "PCE_component_product_kernel"),
    ]:
        print(f"  tuning {model_name}...")
        search = make_block_kernel_search(combination, split_seed)
        search.fit(X_train, y_train)
        pred = search.predict(X_test)

        row = {
            "model": model_name,
            "combination": combination,
            "split_seed": split_seed,
            "inner_cv_splits": INNER_CV_SPLITS,
            "n_iter_search": search.n_iter,
            "best_inner_cv_RMSE": float(-search.best_score_),
            "n_train": len(X_train),
            "n_test": len(X_test),
            "n_union_features": len(PCE_SELECTED_FEATURES),
            "n_jsc_features": len(BLOCK_FEATURES["Jsc"]),
            "n_voc_features": len(BLOCK_FEATURES["Voc"]),
            "n_ff_features": len(BLOCK_FEATURES["FF"]),
            "PCE_R2": float(r2_score(y_test, pred)),
            "PCE_RMSE": rmse_score(y_test, pred),
            "best_params": search.best_params_,
        }
        kernel_rows.append(row)

        print(
            f"    {model_name:<30} | inner RMSE={row['best_inner_cv_RMSE']:.4f} "
            f"| test R2={row['PCE_R2']:.4f} | test RMSE={row['PCE_RMSE']:.4f}"
        )

kernel_results = pd.DataFrame(kernel_rows)
kernel_results


PCE custom block kernel | held-out split seed 0
  tuning PCE_component_additive_kernel...
    PCE_component_additive_kernel  | inner RMSE=2.9090 | test R2=0.6870 | test RMSE=2.6898
  tuning PCE_component_product_kernel...
    PCE_component_product_kernel   | inner RMSE=3.1098 | test R2=0.6293 | test RMSE=2.9270

PCE custom block kernel | held-out split seed 1
  tuning PCE_component_additive_kernel...
    PCE_component_additive_kernel  | inner RMSE=2.9441 | test R2=0.6741 | test RMSE=2.7036
  tuning PCE_component_product_kernel...
    PCE_component_product_kernel   | inner RMSE=2.9388 | test R2=0.6770 | test RMSE=2.6916

PCE custom block kernel | held-out split seed 2
  tuning PCE_component_additive_kernel...
    PCE_component_additive_kernel  | inner RMSE=2.9500 | test R2=0.7172 | test RMSE=2.5866
  tuning PCE_component_product_kernel...
    PCE_component_product_kernel   | inner RMSE=3.0681 | test R2=0.6659 | test RMSE=2.8116

PCE custom block kernel | held-out split seed 3
  tuning 

,model,combination,split_seed,inner_cv_splits,n_iter_search,best_inner_cv_RMSE,n_train,n_test,n_union_features,n_jsc_features,n_voc_features,n_ff_features,PCE_R2,PCE_RMSE,best_params
0,PCE_component_additive_kernel,additive,0,3,16,2.909048,2545,637,37,20,20,30,0.686961,2.689802,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 3.0,..."
1,PCE_component_product_kernel,product,0,3,16,3.109755,2545,637,37,20,20,30,0.629319,2.926987,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 3.0,..."
2,PCE_component_additive_kernel,additive,1,3,16,2.944147,2545,637,37,20,20,30,0.674129,2.703603,"{'gamma_voc_mult': 3.0, 'gamma_jsc_mult': 0.3,..."
3,PCE_component_product_kernel,product,1,3,16,2.938768,2545,637,37,20,20,30,0.677021,2.691578,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 1.0,..."
4,PCE_component_additive_kernel,additive,2,3,16,2.950022,2545,637,37,20,20,30,0.717241,2.586646,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 10.0..."
5,PCE_component_product_kernel,product,2,3,16,3.068067,2545,637,37,20,20,30,0.665923,2.811593,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 1.0,..."
6,PCE_component_additive_kernel,additive,3,3,16,2.837826,2545,637,37,20,20,30,0.624343,2.926125,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 1.0,..."
7,PCE_component_product_kernel,product,3,3,16,2.848216,2545,637,37,20,20,30,0.617969,2.950845,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 0.3,..."
8,PCE_component_additive_kernel,additive,4,3,16,2.927396,2545,637,37,20,20,30,0.717137,2.562253,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 1.0,..."
9,PCE_component_product_kernel,product,4,3,16,3.043528,2545,637,37,20,20,30,0.701010,2.634281,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 0.3,..."


In [10]:
# ============================================================
# 9. Summary: additive vs product block kernel
# ============================================================

kernel_summary = (
    kernel_results
    .groupby(["model", "combination"])
    .agg(
        n_splits=("split_seed", "nunique"),
        PCE_R2_mean=("PCE_R2", "mean"),
        PCE_R2_std=("PCE_R2", "std"),
        PCE_R2_min=("PCE_R2", "min"),
        PCE_R2_max=("PCE_R2", "max"),
        PCE_RMSE_mean=("PCE_RMSE", "mean"),
        PCE_RMSE_std=("PCE_RMSE", "std"),
        best_inner_cv_RMSE_mean=("best_inner_cv_RMSE", "mean"),
        n_union_features=("n_union_features", "mean"),
        n_jsc_features=("n_jsc_features", "mean"),
        n_voc_features=("n_voc_features", "mean"),
        n_ff_features=("n_ff_features", "mean"),
    )
    .reset_index()
    .sort_values("PCE_R2_mean", ascending=False)
)

display(kernel_summary)

kernel_best_params = kernel_results[["model", "split_seed", "best_params"]].copy()
display(kernel_best_params)

,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_R2_min,PCE_R2_max,PCE_RMSE_mean,PCE_RMSE_std,best_inner_cv_RMSE_mean,n_union_features,n_jsc_features,n_voc_features,n_ff_features
0,PCE_component_additive_kernel,additive,10,0.685984,0.031032,0.624343,0.72347,2.670278,0.123441,2.912485,37.0,20.0,20.0,30.0
1,PCE_component_product_kernel,product,10,0.662144,0.026057,0.617969,0.70101,2.771484,0.113432,3.015378,37.0,20.0,20.0,30.0


,model,split_seed,best_params
0,PCE_component_additive_kernel,0,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 3.0,..."
1,PCE_component_product_kernel,0,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 3.0,..."
2,PCE_component_additive_kernel,1,"{'gamma_voc_mult': 3.0, 'gamma_jsc_mult': 0.3,..."
3,PCE_component_product_kernel,1,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 1.0,..."
4,PCE_component_additive_kernel,2,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 10.0..."
5,PCE_component_product_kernel,2,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 1.0,..."
6,PCE_component_additive_kernel,3,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 1.0,..."
7,PCE_component_product_kernel,3,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 0.3,..."
8,PCE_component_additive_kernel,4,"{'gamma_voc_mult': 0.3, 'gamma_jsc_mult': 1.0,..."
9,PCE_component_product_kernel,4,"{'gamma_voc_mult': 1.0, 'gamma_jsc_mult': 0.3,..."


In [ ]:
# ============================================================
# 10. Explicitly audit all 37 selected features by framework level
# ============================================================

FEATURE_LEVEL_CATALOG = {
    # --------------------------------------------------------
    # Level 1 — material and material-stack identity
    # --------------------------------------------------------
    "perovskite_bandgap_clean": (
        "Level 1", "Intrinsic material property", "absorber bandgap"
    ),
    "perovskite.composition_short_form": (
        "Level 1", "Absorber composition", "perovskite composition identity"
    ),
    "perovskite.composition_leadfree": (
        "Level 1", "Absorber composition", "lead-free composition flag"
    ),
    "perovskite.composition_a_ions": (
        "Level 1", "Absorber composition", "A-site ion identity"
    ),
    "perovskite.composition_b_ions": (
        "Level 1", "Absorber composition", "B-site ion identity"
    ),
    "perovskite.composition_c_ions": (
        "Level 1", "Absorber composition", "X/C-site ion identity"
    ),
    "perovskite.composition_a_ions_coefficients": (
        "Level 1", "Absorber composition", "A-site stoichiometric coefficients"
    ),
    "perovskite.composition_c_ions_coefficients": (
        "Level 1", "Absorber composition", "X/C-site stoichiometric coefficients"
    ),
    "etl.additives_compounds": (
        "Level 1", "Transport material identity", "ETL additive/compound identity"
    ),
    "htl.stack_sequence": (
        "Level 1", "Layer-specific material stack", "HTL material-stack identity"
    ),
    "etl.stack_sequence": (
        "Level 1", "Layer-specific material stack", "ETL material-stack identity"
    ),
    "backcontact.stack_sequence": (
        "Level 1", "Layer-specific material stack", "back-contact material-stack identity"
    ),
    "encapsulation.stack_sequence": (
        "Level 1", "Layer-specific material stack", "encapsulation material-stack identity"
    ),
    "substrate.stack_sequence": (
        "Level 1", "Layer-specific material stack", "substrate/front-electrode material-stack identity"
    ),
    "cell.stack_sequence": (
        "Level 1",
        "Aggregate material stack",
        "global layer order and cross-layer material combination; hierarchically overlaps with layer-specific stacks",
    ),

    # --------------------------------------------------------
    # Level 2 — device, process, and measurement
    # --------------------------------------------------------
    "PVK_thickness_clean": (
        "Level 2", "Device geometry/status", "absorber thickness"
    ),
    "ETL_thickness_clean": (
        "Level 2", "Device geometry/status", "ETL thickness"
    ),
    "HTL_thickness_clean": (
        "Level 2", "Device geometry/status", "HTL thickness"
    ),
    "BC_thickness_clean": (
        "Level 2", "Device geometry/status", "back-contact thickness"
    ),
    "ETL_n_layers": (
        "Level 2", "Device geometry/status", "number of ETL layers"
    ),
    "cell.area_measured": (
        "Level 2", "Device/measurement geometry", "measured device area"
    ),
    "etl.deposition_procedure": (
        "Level 2", "Fabrication process", "ETL deposition procedure"
    ),
    "etl.deposition_thermal_annealing_temperature": (
        "Level 2", "Fabrication process", "ETL thermal annealing temperature"
    ),
    "htl.deposition_procedure": (
        "Level 2", "Fabrication process", "HTL deposition procedure"
    ),
    "perovskite_deposition.procedure": (
        "Level 2", "Fabrication process", "perovskite deposition procedure"
    ),
    "perovskite_deposition.solvents": (
        "Level 2", "Fabrication process", "perovskite precursor solvents"
    ),
    "perovskite_deposition.solvents_mixing_ratios": (
        "Level 2", "Fabrication process", "solvent mixing ratios"
    ),
    "perovskite_deposition.synthesis_atmosphere": (
        "Level 2", "Fabrication process", "perovskite synthesis atmosphere"
    ),
    "perovskite_deposition.thermal_annealing_time": (
        "Level 2", "Fabrication process", "annealing time"
    ),
    "perovskite_deposition.thermal_annealing_temperature": (
        "Level 2", "Fabrication process", "annealing temperature"
    ),
    "perovskite_deposition.quenching_media": (
        "Level 2", "Fabrication process", "quenching medium"
    ),
    "perovskite_deposition.quenching_induced_crystallisation": (
        "Level 2", "Fabrication process", "quenching-induced crystallisation flag"
    ),
    "jv.light_spectra": (
        "Level 2", "Measurement metadata", "JV illumination spectrum"
    ),
    "jv.light_intensity": (
        "Level 2", "Measurement metadata", "JV illumination intensity"
    ),
    "jv.test_atmosphere": (
        "Level 2", "Measurement metadata", "JV test atmosphere"
    ),
    "jv.light_masked_cell": (
        "Level 2", "Measurement metadata", "masked-cell status during JV measurement"
    ),
    "jv.average_over_n_number_of_cells": (
        "Level 2", "Measurement metadata", "number of cells averaged for the reported JV value"
    ),
    "jv.default_PCE_scan_direction": (
        "Level 2", "Measurement metadata", "PCE scan direction"
    ),
    "eqe.measured": (
        "Level 2", "Measurement metadata", "EQE measurement availability"
    ),
    "stabilised.performance_measured": (
        "Level 2", "Measurement metadata", "stabilised-performance measurement availability"
    ),
    "stability.average_over_n_number_of_cells": (
        "Level 2", "Measurement metadata", "number of cells averaged in stability reporting"
    ),
    "stability.measured": (
        "Level 2", "Measurement metadata", "stability measurement availability"
    ),
    "stability.atmosphere": (
        "Level 2", "Measurement metadata", "stability-test atmosphere"
    ),

    # --------------------------------------------------------
    # Level 3 — engineered physical descriptors (2)
    # --------------------------------------------------------
    "Absorber_to_ETL_ratio": (
        "Level 3", "Engineered ratio", "PVK thickness divided by ETL thickness"
    ),
    "ff_PVK_fraction_of_total_stack": (
        "Level 3", "Engineered ratio", "PVK thickness fraction of total recorded stack thickness"
    ),
}

missing_from_catalog = sorted(
    set(PCE_SELECTED_FEATURES) - set(FEATURE_LEVEL_CATALOG)
)
assert not missing_from_catalog, (
    "Selected features missing from framework catalogue: "
    + ", ".join(missing_from_catalog)
)
FEATURE_LEVEL_MANIFEST = {
    feature: FEATURE_LEVEL_CATALOG[feature]
    for feature in PCE_SELECTED_FEATURES
}


def membership_label(feature):
    blocks = []
    if feature in JSC_SELECTED_PRESENT:
        blocks.append("Jsc")
    if feature in VOC_SELECTED_PRESENT:
        blocks.append("Voc")
    if feature in FF_SELECTED_PRESENT:
        blocks.append("FF")
    return "+".join(blocks)


manifest_features = set(FEATURE_LEVEL_MANIFEST)
selected_features = set(PCE_SELECTED_FEATURES)
missing_from_manifest = sorted(selected_features - manifest_features)
extra_in_manifest = sorted(manifest_features - selected_features)

assert not missing_from_manifest, (
    "Selected features missing from framework manifest: "
    + ", ".join(missing_from_manifest)
)
assert not extra_in_manifest, (
    "Framework manifest contains non-selected features: "
    + ", ".join(extra_in_manifest)
)
assert len(FEATURE_LEVEL_MANIFEST) == len(PCE_SELECTED_FEATURES) == EXPECTED_PCE_UNION_COUNT

feature_grouping_rows = []
for feature in PCE_SELECTED_FEATURES:
    level, physical_group, rationale = FEATURE_LEVEL_MANIFEST[feature]
    feature_grouping_rows.append({
        "feature": feature,
        "component_membership": membership_label(feature),
        "framework_level": level,
        "physical_group": physical_group,
        "grouping_rationale": rationale,
    })

pce_feature_grouping = pd.DataFrame(feature_grouping_rows)

level_order = {"Level 1": 1, "Level 2": 2, "Level 3": 3}
pce_feature_grouping = (
    pce_feature_grouping
    .assign(_level_order=lambda d: d["framework_level"].map(level_order))
    .sort_values(["_level_order", "physical_group", "feature"])
    .drop(columns="_level_order")
    .reset_index(drop=True)
)

display(pce_feature_grouping)

feature_grouping_summary = (
    pce_feature_grouping
    .groupby(["framework_level", "physical_group"], as_index=False)
    .agg(
        n_features=("feature", "count"),
        features=("feature", lambda values: "; ".join(values)),
    )
    .assign(_level_order=lambda d: d["framework_level"].map(level_order))
    .sort_values(["_level_order", "physical_group"])
    .drop(columns="_level_order")
    .reset_index(drop=True)
)

display(feature_grouping_summary)
print(
    "Feature counts by framework level:",
    pce_feature_grouping["framework_level"].value_counts().to_dict(),
)


,feature,component_membership,framework_level,physical_group,grouping_rationale
0,perovskite.composition_a_ions,Voc,Level 1,Absorber composition,A-site ion identity
1,perovskite.composition_b_ions,Voc,Level 1,Absorber composition,B-site ion identity
2,perovskite.composition_c_ions,Voc,Level 1,Absorber composition,X/C-site ion identity
3,perovskite.composition_c_ions_coefficients,FF,Level 1,Absorber composition,X/C-site stoichiometric coefficients
4,perovskite.composition_leadfree,Voc+FF,Level 1,Absorber composition,lead-free composition flag
5,perovskite.composition_short_form,Jsc,Level 1,Absorber composition,perovskite composition identity
6,cell.stack_sequence,FF,Level 1,Aggregate material stack,global layer order and cross-layer material co...
7,perovskite_bandgap_clean,Jsc+Voc+FF,Level 1,Intrinsic material property,absorber bandgap
8,backcontact.stack_sequence,Jsc+FF,Level 1,Layer-specific material stack,back-contact material-stack identity
9,etl.stack_sequence,Jsc+Voc+FF,Level 1,Layer-specific material stack,ETL material-stack identity


,framework_level,physical_group,n_features,features
0,Level 1,Absorber composition,6,perovskite.composition_a_ions; perovskite.comp...
1,Level 1,Aggregate material stack,1,cell.stack_sequence
2,Level 1,Intrinsic material property,1,perovskite_bandgap_clean
3,Level 1,Layer-specific material stack,3,backcontact.stack_sequence; etl.stack_sequence...
4,Level 1,Transport material identity,1,etl.additives_compounds
5,Level 2,Device geometry/status,5,BC_thickness_clean; ETL_n_layers; ETL_thicknes...
6,Level 2,Device/measurement geometry,1,cell.area_measured
7,Level 2,Fabrication process,10,etl.deposition_procedure; htl.deposition_proce...
8,Level 2,Measurement metadata,7,eqe.measured; jv.average_over_n_number_of_cell...
9,Level 3,Engineered ratio,2,Absorber_to_ETL_ratio; ff_PVK_fraction_of_tota...


Feature counts by framework level: {'Level 2': 23, 'Level 1': 12, 'Level 3': 2}


In [12]:
# ============================================================
# 11. Define the audited Level 1/2/3 feature blocks
# ============================================================

LEVEL1_MATERIAL_FEATURES = pce_feature_grouping.loc[
    pce_feature_grouping["framework_level"].eq("Level 1"), "feature"
].tolist()

LEVEL2_DEVICE_PROCESS_MEASUREMENT_FEATURES = pce_feature_grouping.loc[
    pce_feature_grouping["framework_level"].eq("Level 2"), "feature"
].tolist()

LEVEL3_DERIVED_FEATURES = pce_feature_grouping.loc[
    pce_feature_grouping["framework_level"].eq("Level 3"), "feature"
].tolist()

FRAMEWORK_BLOCK_FEATURES = {
    "Level1_material_stack": LEVEL1_MATERIAL_FEATURES,
    "Level2_device_process_measurement": LEVEL2_DEVICE_PROCESS_MEASUREMENT_FEATURES,
    "Level3_derived": LEVEL3_DERIVED_FEATURES,
}

EXPECTED_LEVEL_COUNTS = {
    "Level1_material_stack": 12,
    "Level2_device_process_measurement": 23,
    "Level3_derived": 2,
}
actual_level_counts = {
    block: len(features)
    for block, features in FRAMEWORK_BLOCK_FEATURES.items()
}
assert actual_level_counts == EXPECTED_LEVEL_COUNTS, (
    f"Expected {EXPECTED_LEVEL_COUNTS}, got {actual_level_counts}"
)

framework_block_audit = pd.DataFrame({
    "kernel_block": list(FRAMEWORK_BLOCK_FEATURES.keys()),
    "n_features": [len(v) for v in FRAMEWORK_BLOCK_FEATURES.values()],
    "features": ["; ".join(v) for v in FRAMEWORK_BLOCK_FEATURES.values()],
})

display(framework_block_audit)
print("Framework block counts:", actual_level_counts)


,kernel_block,n_features,features
0,Level1_material_stack,12,perovskite.composition_a_ions; perovskite.comp...
1,Level2_device_process_measurement,23,BC_thickness_clean; ETL_n_layers; ETL_thicknes...
2,Level3_derived,2,Absorber_to_ETL_ratio; ff_PVK_fraction_of_tota...


Framework block counts: {'Level1_material_stack': 12, 'Level2_device_process_measurement': 23, 'Level3_derived': 2}


In [13]:
# ============================================================
# PCE custom Level block kernels
# Compare:
#   1. K_L1 + K_L2 + K_L3
#   2. K_L1 * K_L2 * K_L3
#   3. pairwise interactions
#   4. (1 + K_L1)(1 + K_L2)(1 + K_L3)
# ============================================================

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, KFold
from sklearn.kernel_ridge import KernelRidge
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.metrics.pairwise import rbf_kernel, euclidean_distances


# ----------------------------
# Settings
# ----------------------------
HOLDOUT_SEEDS_LEVEL_KERNEL = list(HOLDOUT_SEEDS)

INNER_SPLITS = 3
ALPHAS = [0.1, 1.0, 10.0, 100.0]
GAMMA_MULTIPLIERS = [0.5, 1.0, 2.0]

LEVEL_KERNEL_COMBINATIONS = [
    "level_additive",
    "level_product",
    "level_pairwise_interaction",
    "level_interaction_augmented",
]


# ----------------------------
# Helpers
# ----------------------------


def rmse_score(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))


def make_block_preprocessor(X_block):
    num_cols = X_block.select_dtypes(include=np.number).columns.tolist()
    cat_cols = [c for c in X_block.columns if c not in num_cols]

    transformers = []

    if num_cols:
        numeric_transformer = Pipeline([
            ("imputer", SimpleImputer(strategy="mean")),
            ("scaler", StandardScaler()),
        ])
        transformers.append(("num", numeric_transformer, num_cols))

    if cat_cols:
        categorical_transformer = Pipeline([
            ("to_object", FunctionTransformer(
                lambda x: x.astype(object), validate=False
            )),
            ("imputer", SimpleImputer(strategy="constant", fill_value="Missing")),
            ("to_string", FunctionTransformer(lambda x: x.astype(str), validate=False)),
            ("onehot", make_onehot_encoder()),
        ])
        transformers.append(("cat", categorical_transformer, cat_cols))

    return ColumnTransformer(transformers)


def median_gamma(Z, max_samples=800, random_state=42):
    n = Z.shape[0]
    rng = np.random.default_rng(random_state)

    if n > max_samples:
        idx = rng.choice(n, size=max_samples, replace=False)
        Z_sample = Z[idx]
    else:
        Z_sample = Z

    D2 = euclidean_distances(Z_sample, Z_sample, squared=True)
    vals = D2[np.triu_indices_from(D2, k=1)]
    vals = vals[np.isfinite(vals)]
    vals = vals[vals > 0]

    if len(vals) == 0:
        return 1.0

    return 1.0 / np.median(vals)


def combine_kernels(block_kernels, combination):
    Ks = list(block_kernels.values())
    n_blocks = len(Ks)

    if combination == "level_additive":
        return sum(Ks) / n_blocks

    if combination == "level_product":
        K = Ks[0].copy()
        for Ki in Ks[1:]:
            K *= Ki
        return K

    if combination == "level_pairwise_interaction":
        pairwise = []
        for i in range(n_blocks):
            for j in range(i + 1, n_blocks):
                pairwise.append(Ks[i] * Ks[j])
        return sum(pairwise) / len(pairwise)

    if combination == "level_interaction_augmented":
        # ((1 + K1)(1 + K2)(1 + K3)) / 2^3
        # keeps diagonal approximately 1, while allowing additive + interaction effects
        K = np.ones_like(Ks[0])
        for Ki in Ks:
            K *= (1.0 + Ki)
        return K / (2 ** n_blocks)

    raise ValueError(f"Unknown combination: {combination}")


def fit_transform_blocks(X_train, X_other, level_feature_blocks, random_state=42, gamma_multiplier=1.0):
    train_kernels = {}
    other_kernels = {}

    for level_name, cols in level_feature_blocks.items():
        if len(cols) == 0:
            continue

        prep = make_block_preprocessor(X_train[cols])
        Z_train = prep.fit_transform(X_train[cols])
        Z_other = prep.transform(X_other[cols])

        gamma = median_gamma(Z_train, random_state=random_state) * gamma_multiplier

        train_kernels[level_name] = rbf_kernel(Z_train, Z_train, gamma=gamma)
        other_kernels[level_name] = rbf_kernel(Z_other, Z_train, gamma=gamma)

    return train_kernels, other_kernels


# ----------------------------
# Manually audited framework blocks
# ----------------------------
# Use the Level-1/2/3 grouping defined in Cells 10-11.
# No keyword-based reclassification is allowed here.
EXPECTED_LEVEL_COUNTS = {
    "Level1_material_stack": 12,
    "Level2_device_process_measurement": 23,
    "Level3_derived": 2,
}

level_feature_blocks = {
    level: list(FRAMEWORK_BLOCK_FEATURES[level])
    for level in EXPECTED_LEVEL_COUNTS
}

all_level_features = [
    feature
    for features in level_feature_blocks.values()
    for feature in features
]

duplicate_level_features = sorted({
    feature
    for feature in all_level_features
    if all_level_features.count(feature) > 1
})
missing_level_features = sorted(set(X_pce_selected.columns) - set(all_level_features))
extra_level_features = sorted(set(all_level_features) - set(X_pce_selected.columns))
actual_level_counts = {
    level: len(features)
    for level, features in level_feature_blocks.items()
}

assert not duplicate_level_features, (
    "Features assigned to more than one framework block: "
    + ", ".join(duplicate_level_features)
)
assert not missing_level_features, (
    "Selected PCE features missing from framework blocks: "
    + ", ".join(missing_level_features)
)
assert not extra_level_features, (
    "Framework blocks contain features outside X_pce_selected: "
    + ", ".join(extra_level_features)
)
assert actual_level_counts == EXPECTED_LEVEL_COUNTS, (
    f"Unexpected framework block sizes. "
    f"Expected {EXPECTED_LEVEL_COUNTS}, got {actual_level_counts}"
)
assert len(all_level_features) == X_pce_selected.shape[1] == EXPECTED_PCE_UNION_COUNT

level_kernel_feature_audit = pd.DataFrame([
    {
        "level_block": level,
        "n_features": len(features),
        "features": "; ".join(features),
    }
    for level, features in level_feature_blocks.items()
])

display(level_kernel_feature_audit)
print("Framework feature audit passed:", actual_level_counts)
print(f"All {len(all_level_features)} selected PCE features are used exactly once.")


# ----------------------------
# Inner CV tuning for one outer split
# ----------------------------
def tune_level_kernel_on_train(X_train, y_train, outer_seed):
    inner_cv = KFold(n_splits=INNER_SPLITS, shuffle=True, random_state=outer_seed)

    records = []

    for combination in LEVEL_KERNEL_COMBINATIONS:
        for gamma_multiplier in GAMMA_MULTIPLIERS:
            for alpha in ALPHAS:
                fold_rmses = []

                for inner_train_idx, inner_valid_idx in inner_cv.split(X_train):
                    X_inner_train = X_train.iloc[inner_train_idx]
                    X_inner_valid = X_train.iloc[inner_valid_idx]

                    y_inner_train = y_train.iloc[inner_train_idx]
                    y_inner_valid = y_train.iloc[inner_valid_idx]

                    inner_train_blocks, inner_valid_blocks = fit_transform_blocks(
                        X_inner_train,
                        X_inner_valid,
                        level_feature_blocks,
                        random_state=outer_seed,
                        gamma_multiplier=gamma_multiplier,
                    )

                    K_inner_train = combine_kernels(inner_train_blocks, combination)
                    K_inner_valid = combine_kernels(inner_valid_blocks, combination)

                    # Precomputed KernelRidge has no explicit intercept.
                    # Centre using the inner-training fold only.
                    y_inner_train_array = np.asarray(y_inner_train, dtype=float)
                    y_inner_mean = float(np.mean(y_inner_train_array))

                    model = KernelRidge(alpha=alpha, kernel="precomputed")
                    model.fit(
                        K_inner_train,
                        y_inner_train_array - y_inner_mean,
                    )

                    pred = model.predict(K_inner_valid) + y_inner_mean
                    fold_rmses.append(rmse_score(y_inner_valid, pred))

                records.append({
                    "combination": combination,
                    "gamma_multiplier": gamma_multiplier,
                    "alpha": alpha,
                    "inner_cv_RMSE": np.mean(fold_rmses),
                    "inner_cv_RMSE_std": np.std(fold_rmses),
                })

    tuning_df = pd.DataFrame(records).sort_values("inner_cv_RMSE").reset_index(drop=True)
    return tuning_df.iloc[0].to_dict(), tuning_df


# ----------------------------
# Repeated held-out evaluation
# ----------------------------
level_kernel_results = []
level_kernel_best_params = []

for seed in HOLDOUT_SEEDS_LEVEL_KERNEL:
    print(f"\n=== Outer seed {seed} ===")

    X_train, X_test, y_train, y_test = train_test_split(
        X_pce_selected,
        y_pce,
        test_size=TEST_SIZE,
        random_state=seed,
        shuffle=True,
    )

    best_params, tuning_df = tune_level_kernel_on_train(X_train, y_train, seed)

    print("Best inner-CV params:")
    print(best_params)

    train_blocks, test_blocks = fit_transform_blocks(
        X_train,
        X_test,
        level_feature_blocks,
        random_state=seed,
        gamma_multiplier=best_params["gamma_multiplier"],
    )

    K_train_all = {
        combination: combine_kernels(train_blocks, combination)
        for combination in LEVEL_KERNEL_COMBINATIONS
    }

    K_test_all = {
        combination: combine_kernels(test_blocks, combination)
        for combination in LEVEL_KERNEL_COMBINATIONS
    }

    # Evaluate all combinations using their own best params, not only global best
    for combination in LEVEL_KERNEL_COMBINATIONS:
        best_combo_row = (
            tuning_df[tuning_df["combination"] == combination]
            .sort_values("inner_cv_RMSE")
            .iloc[0]
        )

        gamma_multiplier = float(best_combo_row["gamma_multiplier"])
        alpha = float(best_combo_row["alpha"])

        level_kernel_best_params.append({
            "seed": seed,
            "combination": combination,
            "alpha": alpha,
            "gamma_multiplier": gamma_multiplier,
            "best_inner_cv_RMSE": float(best_combo_row["inner_cv_RMSE"]),
            "best_inner_cv_RMSE_std": float(best_combo_row["inner_cv_RMSE_std"]),
        })

        # recompute if gamma differs from global best
        if gamma_multiplier != best_params["gamma_multiplier"]:
            train_blocks_combo, test_blocks_combo = fit_transform_blocks(
                X_train,
                X_test,
                level_feature_blocks,
                random_state=seed,
                gamma_multiplier=gamma_multiplier,
            )
            K_train = combine_kernels(train_blocks_combo, combination)
            K_test = combine_kernels(test_blocks_combo, combination)
        else:
            K_train = K_train_all[combination]
            K_test = K_test_all[combination]

        y_train_array = np.asarray(y_train, dtype=float)
        y_train_mean = float(np.mean(y_train_array))

        model = KernelRidge(alpha=alpha, kernel="precomputed")
        model.fit(K_train, y_train_array - y_train_mean)

        pred = model.predict(K_test) + y_train_mean

        level_kernel_results.append({
            "seed": seed,
            "model": f"PCE_{combination}_kernel",
            "combination": combination,
            "PCE_R2": r2_score(y_test, pred),
            "PCE_RMSE": rmse_score(y_test, pred),
            "best_inner_cv_RMSE": best_combo_row["inner_cv_RMSE"],
            "best_inner_cv_RMSE_std": best_combo_row["inner_cv_RMSE_std"],
            "alpha": alpha,
            "gamma_multiplier": gamma_multiplier,
            "n_union_features": len(all_level_features),
            "n_level1_features": len(level_feature_blocks["Level1_material_stack"]),
            "n_level2_features": len(level_feature_blocks["Level2_device_process_measurement"]),
            "n_level3_features": len(level_feature_blocks["Level3_derived"]),
        })

    current = pd.DataFrame(level_kernel_results)
    display(
        current
        .groupby(["model", "combination"], as_index=False)
        .agg(
            n_splits=("seed", "nunique"),
            PCE_R2_mean=("PCE_R2", "mean"),
            PCE_R2_std=("PCE_R2", "std"),
            PCE_RMSE_mean=("PCE_RMSE", "mean"),
            PCE_RMSE_std=("PCE_RMSE", "std"),
        )
        .sort_values("PCE_R2_mean", ascending=False)
    )


level_kernel_results_df = pd.DataFrame(level_kernel_results)
level_kernel_best_params_df = pd.DataFrame(level_kernel_best_params)

level_kernel_summary = (
    level_kernel_results_df
    .groupby(["model", "combination"], as_index=False)
    .agg(
        n_splits=("seed", "nunique"),
        PCE_R2_mean=("PCE_R2", "mean"),
        PCE_R2_std=("PCE_R2", "std"),
        PCE_R2_min=("PCE_R2", "min"),
        PCE_R2_max=("PCE_R2", "max"),
        PCE_RMSE_mean=("PCE_RMSE", "mean"),
        PCE_RMSE_std=("PCE_RMSE", "std"),
        best_inner_cv_RMSE_mean=("best_inner_cv_RMSE", "mean"),
        n_union_features=("n_union_features", "mean"),
        n_level1_features=("n_level1_features", "mean"),
        n_level2_features=("n_level2_features", "mean"),
        n_level3_features=("n_level3_features", "mean"),
    )
    .sort_values("PCE_R2_mean", ascending=False)
    .reset_index(drop=True)
)

display(level_kernel_summary)
display(
    level_kernel_best_params_df
    .sort_values(["seed", "combination"])
    .reset_index(drop=True)
)


,level_block,n_features,features
0,Level1_material_stack,12,perovskite.composition_a_ions; perovskite.comp...
1,Level2_device_process_measurement,23,BC_thickness_clean; ETL_n_layers; ETL_thicknes...
2,Level3_derived,2,Absorber_to_ETL_ratio; ff_PVK_fraction_of_tota...


Framework feature audit passed: {'Level1_material_stack': 12, 'Level2_device_process_measurement': 23, 'Level3_derived': 2}
All 37 selected PCE features are used exactly once.

=== Outer seed 0 ===
Best inner-CV params:
{'combination': 'level_interaction_augmented', 'gamma_multiplier': 0.5, 'alpha': 0.1, 'inner_cv_RMSE': 2.9009568348503905, 'inner_cv_RMSE_std': 0.05073083288746869}


,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_RMSE_mean,PCE_RMSE_std
2,PCE_level_pairwise_interaction_kernel,level_pairwise_interaction,1,0.698054,NaN,2.641712,NaN
3,PCE_level_product_kernel,level_product,1,0.696056,NaN,2.650437,NaN
1,PCE_level_interaction_augmented_kernel,level_interaction_augmented,1,0.692857,NaN,2.664348,NaN
0,PCE_level_additive_kernel,level_additive,1,0.683290,NaN,2.705527,NaN



=== Outer seed 1 ===
Best inner-CV params:
{'combination': 'level_interaction_augmented', 'gamma_multiplier': 1.0, 'alpha': 0.1, 'inner_cv_RMSE': 2.9530293211202507, 'inner_cv_RMSE_std': 0.038294350767698694}


,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_RMSE_mean,PCE_RMSE_std
2,PCE_level_pairwise_interaction_kernel,level_pairwise_interaction,2,0.694324,0.005275,2.638065,0.005158
3,PCE_level_product_kernel,level_product,2,0.691189,0.006883,2.651489,0.001488
1,PCE_level_interaction_augmented_kernel,level_interaction_augmented,2,0.690714,0.003031,2.653683,0.015082
0,PCE_level_additive_kernel,level_additive,2,0.679325,0.005607,2.702010,0.004974


...

,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_RMSE_mean,PCE_RMSE_std
3,PCE_level_product_kernel,level_product,9,0.685658,0.036091,2.674330,0.146223
2,PCE_level_pairwise_interaction_kernel,level_pairwise_interaction,9,0.685299,0.035217,2.676136,0.143530
1,PCE_level_interaction_augmented_kernel,level_interaction_augmented,9,0.684887,0.031355,2.678678,0.129089
0,PCE_level_additive_kernel,level_additive,9,0.677849,0.028434,2.708787,0.109560



=== Outer seed 9 ===
Best inner-CV params:
{'combination': 'level_interaction_augmented', 'gamma_multiplier': 1.0, 'alpha': 0.1, 'inner_cv_RMSE': 2.840299154362134, 'inner_cv_RMSE_std': 0.08115984035026495}


,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_RMSE_mean,PCE_RMSE_std
3,PCE_level_product_kernel,level_product,10,0.686314,0.034090,2.668454,0.139106
2,PCE_level_pairwise_interaction_kernel,level_pairwise_interaction,10,0.685832,0.033245,2.670756,0.136387
1,PCE_level_interaction_augmented_kernel,level_interaction_augmented,10,0.685651,0.029660,2.672239,0.123398
0,PCE_level_additive_kernel,level_additive,10,0.678948,0.027033,2.700900,0.106263


,model,combination,n_splits,PCE_R2_mean,PCE_R2_std,PCE_R2_min,PCE_R2_max,PCE_RMSE_mean,PCE_RMSE_std,best_inner_cv_RMSE_mean,n_union_features,n_level1_features,n_level2_features,n_level3_features
0,PCE_level_product_kernel,level_product,10,0.686314,0.034090,0.611450,0.733205,2.668454,0.139106,2.922265,37.0,12.0,23.0,2.0
1,PCE_level_pairwise_interaction_kernel,level_pairwise_interaction,10,0.685832,0.033245,0.612599,0.733547,2.670756,0.136387,2.903416,37.0,12.0,23.0,2.0
2,PCE_level_interaction_augmented_kernel,level_interaction_augmented,10,0.685651,0.029660,0.621741,0.731757,2.672239,0.123398,2.896589,37.0,12.0,23.0,2.0
3,PCE_level_additive_kernel,level_additive,10,0.678948,0.027033,0.631549,0.712848,2.700900,0.106263,2.928701,37.0,12.0,23.0,2.0


,seed,combination,alpha,gamma_multiplier,best_inner_cv_RMSE,best_inner_cv_RMSE_std
0,0,level_additive,0.1,1.0,2.930603,0.033107
1,0,level_interaction_augmented,0.1,0.5,2.900957,0.050731
2,0,level_pairwise_interaction,0.1,0.5,2.907269,0.048070
3,0,level_product,0.1,0.5,2.920363,0.069829
4,1,level_additive,0.1,1.0,2.973291,0.044877
5,1,level_interaction_augmented,0.1,1.0,2.953029,0.038294
6,1,level_pairwise_interaction,0.1,0.5,2.956870,0.033389
7,1,level_product,0.1,0.5,2.965020,0.032844
8,2,level_additive,0.1,1.0,2.961097,0.076074
9,2,level_interaction_augmented,0.1,0.5,2.938995,0.084955
